# Writing Tools NLP — QLoRA fine-tuning (Colab T4)

Fine-tune **Qwen3-4B-Instruct-2507** with QLoRA on ~4k writing-task examples.

**Preferred path:** use [`notebooks/train_kaggle.ipynb`](train_kaggle.ipynb) — Kaggle free GPU sessions last up to ~9h, enough for a full 4B epoch.

**Colab note:** free Colab sessions (~2–3h) are usually too short for 4B. If you use Colab, mount Google Drive (Cell 2) and resume across sessions.

**Runtime:** GPU → T4. Do not run on Mac — `bitsandbytes` is CUDA-only.

**Steps:** clone repo → (optional Drive checkpoints) → prepare data → train → eval → download `adapter/`.

Mid-training eval is disabled in `train/train.py` to avoid OOM; run `train/eval.py` after training (see Cell 6).

In [ ]:
# Cell 1 — GPU check + clone repo
import torch

assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → T4"
print(f"GPU: {torch.cuda.get_device_name(0)}")

# Clone (replace with your fork URL if needed)
!git clone https://github.com/Aditya-ice/writing-tools-NLP.git
%cd writing-tools-NLP

In [ ]:
# Cell 2 — Optional: persist checkpoints to Google Drive (recommended for Colab)
# Skip this cell on Kaggle.

# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/writing-tools-checkpoints
# !ln -sfn /content/drive/MyDrive/writing-tools-checkpoints checkpoints

In [ ]:
# Cell 2 — Install dependencies (CUDA build of torch is preinstalled on Colab)
%pip install -q -r requirements.txt

# mlx is Apple-Silicon-only; if present on Colab its broken Linux wheel crashes
# transformers mid-training (libmlx.so ImportError). Make sure it is gone.
%pip uninstall -y -q mlx mlx-lm

In [ ]:
# Cell 3 — Prepare datasets (~4k examples, ~30s)
!python data/prepare.py

In [ ]:
# Cell 5 — Train QLoRA adapter (~6–8h for full 4B epoch; Colab free tier may need resume)
# Resume after a disconnect:
#   !python train/train.py --resume_from_checkpoint ./checkpoints/checkpoint-XXX

!python train/train.py

In [ ]:
# Cell 6 — ROUGE + BERTScore eval (base vs fine-tuned)
# Quick smoke test: !python train/eval.py --adapter_path ./adapter --split val --max_samples 50

!python train/eval.py --adapter_path ./adapter --split val

In [ ]:
# Cell 7 — Download adapter weights
from google.colab import files
import shutil

shutil.make_archive("adapter", "zip", "adapter")
files.download("adapter.zip")